In [3]:
!pip install --upgrade transformers peft

In [2]:
"""
Phase 3: GRPO post-training for mathematical reasoning emergence

Implements Group Relative Policy Optimization (GRPO) as used in DeepSeek-R1:
- Sample G completions per prompt (group)
- Compute verifiable rewards (exact answer match + format)
- Normalize rewards within the group to get advantages
- Update policy with clipped surrogate objective + KL penalty

Key design choices:
- No learned critic / value network (unlike PPO)
- KL penalty against frozen reference model
- Composite reward: correctness (1.0) + format (0.3) + penalties
- LoRA-only training for memory efficiency (merge before Phase eval)
"""
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json
import math
import random
import argparse
import time
from pathlib import Path
from typing import Optional
from dataclasses import dataclass, asdict

import torch
import torch.nn.functional as F
import numpy as np
import wandb
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk

In [4]:
# Remove the conflicting 'torch' directory
!rm -rf /root/.ipython/torch
print("Conflicting '/root/.ipython/torch' directory removed. Please restart the Colab runtime now.")


Conflicting '/root/.ipython/torch' directory removed. Please restart the Colab runtime now.


Let's check your `sys.path` for any potentially conflicting `torch` modules that might be shadowing the official PyTorch library. If any are found, you might need to rename or delete them.

In [5]:
import sys
import os

print("Checking sys.path for conflicting 'torch' modules:")
found_conflict = False

# Try to get the path of the actual installed torch module, if it's loaded
real_torch_path = None
try:
    if 'torch' in sys.modules:
        real_torch_path = os.path.dirname(sys.modules['torch'].__file__)
except AttributeError: # Handle cases where torch might be partially initialized
    pass

for path in sys.path:
    potential_torch_file = os.path.join(path, 'torch.py')
    potential_torch_dir = os.path.join(path, 'torch')

    # Flag any torch.py file or torch directory as a potential conflict
    # unless it's the actual installed torch package path.
    if os.path.exists(potential_torch_file):
        print(f"  Found potential conflicting file: {potential_torch_file}")
        found_conflict = True
    if os.path.isdir(potential_torch_dir) and potential_torch_dir != real_torch_path:
        print(f"  Found potential conflicting directory: {potential_torch_dir}")
        found_conflict = True

if not found_conflict:
    print("  No obvious conflicting 'torch.py' files or 'torch' directories found in sys.path (excluding the official torch installation).")
    print("  If the 'torch' import error persists, consider restarting the Colab runtime and reinstalling PyTorch or any recently updated libraries like 'torchao'.")
else:
    print("  The above paths are potential conflicts. Please consider renaming or removing them and then restart the runtime.")


Checking sys.path for conflicting 'torch' modules:
  No obvious conflicting 'torch.py' files or 'torch' directories found in sys.path (excluding the official torch installation).
  If the 'torch' import error persists, consider restarting the Colab runtime and reinstalling PyTorch or any recently updated libraries like 'torchao'.


In [6]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [7]:
import re
import unicodedata
from typing import Optional
from fractions import Fraction
from dataclasses import dataclass, field

try:
  from sympy import sympify, simplify
  from sympy.parsing.latex import parse_latex
  SYMPY_AVAILABLE = True
except ImportError:
  SYMPY_AVAILABLE = False

In [8]:
# Reward functions for GRPO post-training
# Composite reward = correctness + format + length penalty
# Each component is independently ablatable via config flags

'''
Design choices:
- Verifiable: rewards are deterministic given (completion, ground_truth)
- Decomposable: each sub-reward is logged separately for analysis
- Hack-resistant: empty <think> and answer repetition are penalized
'''

# Reward config
@dataclass
class RewardConfig:
    """
    Controls which reward components are active and their weights.
    Ablate individual components by setting weights to 0.
    """
    # Correctness
    correctness_weight: float = 1.0 # main signal: exact answer match

    # Format rewards
    think_open_weight: float = 0.1 # reward for <think> tag present
    think_close_weight: float = 0.1 # reward for </think> tag present
    answer_format_weight: float = 0.1 # reward for \\boxed{} or "The answer is"

    # Penalties
    empty_think_penalty: float = -0.5 # penalize <think></think> with < 10 words
    repetition_penalty: float = -0.2 # penalize repeated token sequences
    length_penalty_weight: float = 0.05 # penalize think blocks > max_think_words
    missing_answer_penalty: float = -0.3 # Penalize if model outputs no answer

    # Length control
    max_think_words: int = 500 # think blocks beyond this get penalized
    min_think_words: int = 10 # below this = "empty think"

    # Numeric tolerance
    numeric_tolerance: float = 1e-6

def extract_answer(text: str) -> Optional[str]:
    """
    Extract the final answer from a model completion.
    Tries multiple patterns in order of reliability.
    """
    # 1. Last \\boxed{} (most reliable for MATH-style)
    matches = re.findall(r'\\boxed\{([^}]+)\}', text)
    if matches:
        return _normalize(matches[-1])

    # 2. After </think> tag
    think_end = re.search(r'</think>\s*(.*)', text, re.DOTALL)
    if think_end:
        remainder = think_end.group(1).strip()
        # Try boxed in remainder
        boxed = re.findall(r'\\boxed\{([^}]+)\}', remainder)
        if boxed:
            return _normalize(boxed[-1])
        # Try "The answer is X"
        m = re.search(
            r'[Tt]he\s+(?:final\s+)?answer\s+is\s*:?\s*([\d,\.\-\/\%\+]+)',
            remainder
        )
        if m:
            return _normalize(m.group(1))
        # First number in remainder
        nums = re.findall(r'-?[\d,]+\.?\d*', remainder)
        if nums:
            return _normalize(nums[0])

    # 3. "#### X" (GSM8K format)
    m = re.search(r'####\s*([\d,\.\-]+)', text)
    if m:
        return _normalize(m.group(1))

    # 4. "The answer is X"
    m = re.search(
        r'[Tt]he\s+(?:final\s+)?answer\s+is\s*:?\s*([\d,\.\-\/\%]+)', text
    )
    if m:
        return _normalize(m.group(1))

    # 5. Last number in the whole completion (fallback)
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    return _normalize(nums[-1]) if nums else None


def extract_gold_answer(raw_gold: str) -> Optional[str]:
    """Extract ground truth answer from GSM8K or MATH format."""
    # GSM8K: #### 42
    m = re.search(r'####\s*([\d,\.\-]+)', raw_gold)
    if m:
        return _normalize(m.group(1))
    # MATH: last \boxed{}
    matches = re.findall(r'\\boxed\{([^}]+)\}', raw_gold)
    if matches:
        return _normalize(matches[-1])
    # Plain number
    nums = re.findall(r'-?[\d,]+\.?\d*', raw_gold)
    return _normalize(nums[-1]) if nums else raw_gold.strip()


def _normalize(s: str) -> str:
    s = str(s).replace(',', '').replace('$', '').strip()
    s = unicodedata.normalize('NFKC', s)
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except:
        return s


# ---------------------------------------------------------------------------
# Answer Equivalence
# ---------------------------------------------------------------------------

def math_equal(pred: Optional[str], gold: Optional[str],
               tol: float = 1e-6) -> bool:
    """
    Check mathematical equivalence between pred and gold.
    Handles: integers, floats, fractions, percentages, LaTeX.
    """
    if pred is None or gold is None:
        return False

    pred, gold = str(pred).strip(), str(gold).strip()

    # 1. Exact string match
    if pred == gold:
        return True

    # 2. Numeric comparison
    p_num = _to_float(pred)
    g_num = _to_float(gold)
    if p_num is not None and g_num is not None:
        denom = abs(g_num) + 1e-8
        return abs(p_num - g_num) / denom < tol

    # 3. Fraction comparison
    p_frac = _to_fraction(pred)
    g_frac = _to_fraction(gold)
    if p_frac is not None and g_frac is not None:
        return p_frac == g_frac

    # 4. Sympy symbolic equality (expensive, last resort)
    if SYMPY_AVAILABLE:
        return _sympy_equal(pred, gold)

    return False


def _to_float(s: str) -> Optional[float]:
    s = s.replace(',', '').strip()
    if s.endswith('%'):
        try:
            return float(s[:-1]) / 100
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None


def _to_fraction(s: str) -> Optional[Fraction]:
    m = re.match(r'^(-?\d+)\s*/\s*(-?\d+)$', s.strip())
    if m:
        try:
            return Fraction(int(m.group(1)), int(m.group(2)))
        except (ValueError, ZeroDivisionError):
            return None
    return None


def _sympy_equal(pred: str, gold: str) -> bool:
    for parse in [sympify, parse_latex]:
        try:
            diff = simplify(parse(pred) - parse(gold))
            return diff == 0
        except Exception:
            continue
    return False


# ---------------------------------------------------------------------------
# Individual Reward Components
# ---------------------------------------------------------------------------

def reward_correctness(completion: str, gold: str,
                        weight: float = 1.0) -> tuple[float, dict]:
    """
    +weight if extracted answer matches gold, else 0.
    Returns (reward, info_dict).
    """
    pred = extract_answer(completion)
    correct = math_equal(pred, gold)
    return weight if correct else 0.0, {
        "correct": correct,
        "pred": pred,
        "gold": gold,
    }


def reward_format(completion: str,
                  think_open_w: float = 0.1,
                  think_close_w: float = 0.1,
                  answer_fmt_w: float = 0.1) -> tuple[float, dict]:
    """
    Partial rewards for structural format compliance.
      +think_open_w   if <think> present
      +think_close_w  if </think> present
      +answer_fmt_w   if \\boxed{} or "The answer is" present after </think>
    """
    has_open  = "<think>" in completion
    has_close = "</think>" in completion

    # Answer format: look after </think> if present
    search_region = completion
    think_end = re.search(r'</think>\s*(.*)', completion, re.DOTALL)
    if think_end:
        search_region = think_end.group(1)

    has_answer_fmt = bool(
        re.search(r'\\boxed\{', search_region)
        or re.search(r'[Tt]he\s+(?:final\s+)?answer\s+is', search_region)
    )

    r = (think_open_w  if has_open      else 0.0) \
      + (think_close_w if has_close     else 0.0) \
      + (answer_fmt_w  if has_answer_fmt else 0.0)

    return r, {
        "has_think_open":  has_open,
        "has_think_close": has_close,
        "has_answer_fmt":  has_answer_fmt,
    }


def reward_length_penalty(completion: str,
                           max_think_words: int = 500,
                           min_think_words: int = 10,
                           empty_penalty: float = -0.3,
                           length_weight: float = 0.05) -> tuple[float, dict]:
    """
    Penalizes two failure modes:
      1. Empty <think>: tags present but content < min_think_words  → empty_penalty
      2. Excessive <think>: content > max_think_words               → proportional penalty
    """
    think_match = re.search(r'<think>(.*?)</think>', completion, re.DOTALL)
    has_think   = think_match is not None
    think_words = len(think_match.group(1).split()) if think_match else 0

    penalty = 0.0
    is_empty    = False
    is_excessive = False

    if has_think:
        if think_words < min_think_words:
            penalty    = empty_penalty
            is_empty   = True
        elif think_words > max_think_words:
            excess      = think_words - max_think_words
            penalty     = -length_weight * math.log1p(excess / 100)
            is_excessive = True

    return penalty, {
        "think_words":    think_words,
        "is_empty_think": is_empty,
        "is_excessive":   is_excessive,
    }


def reward_repetition_penalty(completion: str,
                               penalty: float = -0.2,
                               window: int = 10,
                               threshold: float = 0.4) -> tuple[float, dict]:
    """
    Penalizes degenerate repetition (e.g. "42 42 42 42").
    Looks at a sliding window over tokens; if uniqueness ratio < threshold → penalty.
    """
    tokens = completion.split()
    if len(tokens) < window * 2:
        return 0.0, {"repetition_detected": False}

    # Check last `window * 2` tokens for repetition
    tail = tokens[-(window * 2):]
    unique_ratio = len(set(tail)) / len(tail)
    detected = unique_ratio < threshold

    return penalty if detected else 0.0, {
        "repetition_detected": detected,
        "tail_unique_ratio":   round(unique_ratio, 3),
    }


# ---------------------------------------------------------------------------
# Composite Reward
# ---------------------------------------------------------------------------

def compute_reward(
    completion: str,
    gold: str,
    config: RewardConfig = None,
) -> tuple[float, dict]:
    """
    Composite reward function for GRPO.

    Returns:
      total_reward (float): scalar reward for this completion
      breakdown (dict):     per-component values for logging
    """
    if config is None:
        config = RewardConfig()

    # 1. Correctness
    r_correct, info_correct = reward_correctness(
        completion, gold, weight=config.correctness_weight
    )

    # Missing response
    pred = extract_answer(completion)
    r_missing = config.missing_answer_penalty if pred is None else 0.0

    # 2. Format
    r_format, info_format = reward_format(
        completion,
        think_open_w=config.think_open_weight,
        think_close_w=config.think_close_weight,
        answer_fmt_w=config.answer_format_weight,
    )

    # 3. Length penalty
    r_length, info_length = reward_length_penalty(
        completion,
        max_think_words=config.max_think_words,
        min_think_words=config.min_think_words,
        empty_penalty=config.empty_think_penalty,
        length_weight=config.length_penalty_weight,
    )

    # 4. Repetition penalty
    r_repeat, info_repeat = reward_repetition_penalty(
        completion, penalty=config.repetition_penalty
    )

    total = r_correct + r_format + r_length + r_repeat + r_missing

    breakdown = {
        # totals
        "reward_total":       round(total, 4),
        "reward_correctness": round(r_correct, 4),
        "reward_format":      round(r_format, 4),
        "reward_length":      round(r_length, 4),
        "reward_repetition":  round(r_repeat, 4),
        "reward_missing_answer": round(r_missing, 4),
        # details
        **info_correct,
        **info_format,
        **info_length,
        **info_repeat,
    }

    return total, breakdown


# ---------------------------------------------------------------------------
# Batch reward (called by GRPOTrainer)
# ---------------------------------------------------------------------------

def batch_reward_fn(
    completions: list[str],
    gold_answers: list[str],
    config: RewardConfig = None,
) -> tuple[list[float], list[dict]]:
    """
    Score a batch of (completion, gold) pairs.
    completions and gold_answers must be the same length.
    """
    if config is None:
        config = RewardConfig()

    rewards, breakdowns = [], []
    for completion, gold in zip(completions, gold_answers):
        r, b = compute_reward(completion, gold, config)
        rewards.append(r)
        breakdowns.append(b)

    return rewards, breakdowns


# ---------------------------------------------------------------------------
# Reward hacking detection utilities
# ---------------------------------------------------------------------------

def detect_hacking(completions: list[str], rewards: list[float]) -> dict:
    """
    Analyze a group of completions for reward hacking signals.
    Call this per-prompt-group during training for monitoring.
    """
    think_lengths = []
    empty_thinks  = 0
    has_think     = 0
    repetitions   = 0

    for c in completions:
        m = re.search(r'<think>(.*?)</think>', c, re.DOTALL)
        tw = len(m.group(1).split()) if m else 0
        think_lengths.append(tw)

        if "<think>" in c and "</think>" in c:
            has_think += 1
            if tw < 10:
                empty_thinks += 1

        tokens = c.split()
        if len(tokens) >= 20:
            tail = tokens[-20:]
            if len(set(tail)) / len(tail) < 0.4:
                repetitions += 1

    n = len(completions)
    return {
        "group_size":           n,
        "mean_reward":          round(sum(rewards) / n, 4),
        "reward_variance":      round(
            sum((r - sum(rewards)/n)**2 for r in rewards) / n, 4
        ),
        "think_tag_rate":       has_think / n,
        "empty_think_rate":     empty_thinks / n,
        "repetition_rate":      repetitions / n,
        "mean_think_words":     sum(think_lengths) / n,
        "all_same_reward":      len(set(round(r, 3) for r in rewards)) == 1,
        "hacking_suspected":    (
            empty_thinks / max(has_think, 1) > 0.5
            or repetitions / n > 0.3
            or len(set(round(r, 3) for r in rewards)) == 1
        ),
    }



In [9]:
# GRPO Dataset preparation for GRPO training
# GRPO only needs (prompt, ground_truth_answer) pairs - No CoT required
# Model generates its own reasoning
"""
Sources:
  - GSM8K train split (7,473 problems)
  - MATH train split (filtered to levels 1-3)
  - Optional: NuminaMath problems not used in SFT
"""
"""
grpo_dataset.py
---------------
Dataset preparation for GRPO training.

GRPO only needs (prompt, ground_truth_answer) pairs —
NO chain-of-thought is required. The model generates its own reasoning.

Sources:
  - GSM8K train split (7,473 problems)
  - MATH train split (filtered to levels 1-3)
  - Optional: NuminaMath problems not used in SFT
"""

import re
import json
import random
import hashlib
from pathlib import Path
from typing import Optional

from datasets import load_dataset, Dataset, concatenate_datasets


# ---------------------------------------------------------------------------
# Prompt Template
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = (
    "You are a math assistant. Solve the problem step by step.\n"
    "You must answer using exactly this format:\n"
    "<think>\n"
    "your reasoning here\n"
    "</think>\n"
    "\\boxed{numeric final answer only}\n"
    "Do not write anything after the boxed answer."
)

'''
def build_grpo_prompt(problem: str, tokenizer=None) -> str:
    """
    Build the GRPO training prompt.
    Uses chat template if available, otherwise manual Qwen format.
    GRPO prompt must NOT include the answer — only the question.
    """
    return (
        "You are a math assistant. Solve the problem step by step.\n"
        "You must answer exactly in this format:\n"
        "<think>\n"
        "reasoning here\n"
        "</think>\n"
        "\\boxed{numeric answer}\n\n"
        f"Problem: {problem}\n"
        "Answer:\n"
    )
'''
def build_grpo_prompt(problem: str, tokenizer=None) -> str:
    return (
        "<|im_start|>system\n"
        "You are a math assistant. Solve the problem step by step. "
        "Finish with the final answer in \\boxed{}.<|im_end|>\n"
        "<|im_start|>user\n"
        f"Solve: {problem}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


# ---------------------------------------------------------------------------
# GSM8K Loader
# ---------------------------------------------------------------------------

def load_gsm8k_grpo(split: str = "train",
                    tokenizer=None,
                    seed: int = 42) -> list[dict]:
    """
    Load GSM8K and format for GRPO.
    Returns list of {"prompt": str, "gold_answer": str, "problem": str, "source": str}
    """
    print("Loading GSM8K...")
    ds = load_dataset("openai/gsm8k", "main", split=split)

    examples = []
    for ex in ds:
        problem = ex["question"]
        raw_answer = ex["answer"]

        # Extract numeric answer from "#### 42" format
        m = re.search(r'####\s*([\d,\.\-]+)', raw_answer)
        gold = m.group(1).replace(',', '').strip() if m else raw_answer.strip()

        prompt = build_grpo_prompt(problem, tokenizer)
        examples.append({
            "prompt":      prompt,
            "gold_answer": gold,
            "problem":     problem,
            "source":      "gsm8k",
        })

    print(f"  GSM8K: {len(examples)} examples")
    return examples


# ---------------------------------------------------------------------------
# MATH Dataset Loader (filtered to levels 1-3)
# ---------------------------------------------------------------------------

def load_math_grpo(max_level: int = 3,
                   tokenizer=None,
                   seed: int = 42) -> list[dict]:
    """
    Load MATH dataset filtered to difficulty levels 1-3 (GSM8K-adjacent).
    Levels 4-5 are competition level — too hard for a 1.5B model to get
    useful reward signal from during early GRPO training.
    """
    print("Loading MATH dataset...")
    try:
        ds = load_dataset("hendrycks/competition_math", split="train")
    except Exception as e:
        print(f"  Warning: could not load MATH dataset ({e}). Skipping.")
        return []

    examples = []
    for ex in ds:
        # Filter by level
        level_str = ex.get("level", "Level 5")
        try:
            level = int(re.search(r'\d+', level_str).group())
        except (AttributeError, ValueError):
            level = 5

        if level > max_level:
            continue

        problem = ex["problem"]
        solution = ex["solution"]

        # Extract \boxed{} answer
        matches = re.findall(r'\\boxed\{([^}]+)\}', solution)
        if not matches:
            continue
        gold = matches[-1].strip()

        prompt = build_grpo_prompt(problem, tokenizer)
        examples.append({
            "prompt":      prompt,
            "gold_answer": gold,
            "problem":     problem,
            "source":      f"math_level{level}",
        })

    print(f"  MATH (levels 1-{max_level}): {len(examples)} examples")
    return examples


# ---------------------------------------------------------------------------
# Deduplication
# ---------------------------------------------------------------------------

def dedup(examples: list[dict]) -> list[dict]:
    """Remove duplicate problems by normalized problem hash."""
    seen = set()
    unique = []
    for ex in examples:
        text = re.sub(r'\s+', ' ', ex["problem"].lower().strip())
        h = hashlib.md5(text.encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(ex)
    removed = len(examples) - len(unique)
    if removed:
        print(f"  Dedup removed {removed} duplicates → {len(unique)} unique")
    return unique


# ---------------------------------------------------------------------------
# Main Dataset Builder
# ---------------------------------------------------------------------------

def build_grpo_dataset(
    tokenizer=None,
    use_gsm8k: bool = True,
    use_math: bool = True,
    math_max_level: int = 3,
    max_examples: Optional[int] = None,
    val_size: int = 200,
    output_dir: Optional[str] = None,
    seed: int = 42,
) -> tuple[Dataset, Dataset]:
    """
    Build and return (train_dataset, val_dataset) for GRPO training.
    Each example has: prompt, gold_answer, problem, source.
    """

    all_examples = []

    if use_gsm8k:
        print("Loading GSM8K...")
        ds = load_dataset("openai/gsm8k", "main", split="train")
        for ex in ds:
            m = re.search(r"####\s*([\d,\.\-]+)", ex["answer"])
            gold = m.group(1).replace(",", "").strip() if m else ex["answer"].strip()
            all_examples.append({
                "prompt":      build_grpo_prompt(ex["question"], tokenizer),
                "gold_answer": gold,
                "problem":     ex["question"],
                "source":      "gsm8k",
            })
        print(f"  GSM8K: {len(all_examples)} examples")

    if use_math:
        # hendrycks/competition_math was removed from HuggingFace via DMCA.
        # EleutherAI/hendrycks_math is the canonical mirror — identical schema,
        # but split into 7 named configs (one per subject) that must be loaded
        # individually and concatenated.
        MATH_CONFIGS = [
            "algebra",
            "counting_and_probability",
            "geometry",
            "intermediate_algebra",
            "number_theory",
            "prealgebra",
            "precalculus",
        ]
        print("Loading MATH dataset (EleutherAI/hendrycks_math)...")
        before = len(all_examples)
        loaded_configs = 0
        for config_name in MATH_CONFIGS:
            try:
                ds = load_dataset(
                    "EleutherAI/hendrycks_math",
                    config_name,
                    split="train",
                )
                for ex in ds:
                    lvl_str = ex.get("level", "Level 5")
                    try:
                        level = int(re.search(r"\d+", lvl_str).group())
                    except Exception:
                        level = 5
                    if level > math_max_level:
                        continue
                    matches = re.findall(r"\\boxed\{([^}]+)\}", ex["solution"])
                    if not matches:
                        continue
                    all_examples.append({
                        "prompt":      build_grpo_prompt(ex["problem"], tokenizer),
                        "gold_answer": matches[-1].strip(),
                        "problem":     ex["problem"],
                        "source":      f"math_{config_name}_level{level}",
                    })
                loaded_configs += 1
            except Exception as e:
                print(f"  Warning: could not load config '{config_name}': {e}")

        added = len(all_examples) - before
        if added > 0:
            print(f"  MATH levels 1-{math_max_level}: {added} examples "
                  f"({loaded_configs}/{len(MATH_CONFIGS)} configs loaded)")
        else:
            print("  Warning: no MATH examples loaded. "
                  "Continuing with GSM8K only.")

    all_examples = dedup(all_examples)
    random.seed(seed)
    random.shuffle(all_examples)
    if max_examples:
        all_examples = all_examples[: max_examples + val_size]

    val_data   = all_examples[:val_size]
    train_data = all_examples[val_size:]
    print(f"\nGRPO dataset — Train: {len(train_data)}, Val: {len(val_data)}")

    train_ds = Dataset.from_list(train_data)
    val_ds   = Dataset.from_list(val_data)

    if output_dir:
        p = Path(output_dir)
        p.mkdir(parents=True, exist_ok=True)
        train_ds.save_to_disk(str(p / "train"))
        val_ds.save_to_disk(str(p / "val"))
        with open(p / "train.jsonl", "w") as f:
            for ex in train_data:
                f.write(json.dumps(ex) + "\n")
        print(f"  Saved to {output_dir}")

    return train_ds, val_ds


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------
'''
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--model",       type=str,  default="Qwen/Qwen2.5-1.5B")
    parser.add_argument("--output_dir",  type=str,  default="./data/grpo")
    parser.add_argument("--max_examples",type=int,  default=None)
    parser.add_argument("--val_size",    type=int,  default=200)
    parser.add_argument("--math_level",  type=int,  default=3)
    parser.add_argument("--no_math",     action="store_true")
    args = parser.parse_args([])

    from transformers import AutoTokenizer
    try:
        tok = AutoTokenizer.from_pretrained(args.model, trust_remote_code=True)
    except Exception:
        tok = None

    build_grpo_dataset(
        tokenizer=tok,
        use_gsm8k=True,
        use_math=not args.no_math,
        math_max_level=args.math_level,
        max_examples=args.max_examples,
        val_size=args.val_size,
        output_dir=args.output_dir,
    )
'''

'\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--model",       type=str,  default="Qwen/Qwen2.5-1.5B")\n    parser.add_argument("--output_dir",  type=str,  default="./data/grpo")\n    parser.add_argument("--max_examples",type=int,  default=None)\n    parser.add_argument("--val_size",    type=int,  default=200)\n    parser.add_argument("--math_level",  type=int,  default=3)\n    parser.add_argument("--no_math",     action="store_true")\n    args = parser.parse_args([])\n\n    from transformers import AutoTokenizer\n    try:\n        tok = AutoTokenizer.from_pretrained(args.model, trust_remote_code=True)\n    except Exception:\n        tok = None\n\n    build_grpo_dataset(\n        tokenizer=tok,\n        use_gsm8k=True,\n        use_math=not args.no_math,\n        math_max_level=args.math_level,\n        max_examples=args.max_examples,\n        val_size=args.val_size,\n        output_dir=args.output_dir,\n    )\n'

In [12]:
# Training Config
@dataclass
class GRPOConfig:
  # Model paths
  sft_checkpoint: str = "./sft_adapter" # Phase 2 LoRA adapter directory output folder
  model_name: str = "Qwen/Qwen2.5-1.5B"
  output_dir: str = "./checkpoints/grpo"

  # GRPO core hyperparameters
  num_generations: int = 3      # G: completions per prompt
  temperature: float   = 0.9    # sampling
  top_p: float         = 0.95
  kl_coeff: float      = 0.02   # beta: KL penalty weight
  clip_eps: float      = 0.2    # PPO-style clipping epsilon
  max_new_tokens: int  = 256

  # Optimization
  learning_rate: float       = 5e-7 # much lower than SFT
  per_device_batch_size: int = 1 # prompts per GPU step
  gradient_accumulation: int = 3 # effective batch = 16 prompts x 6 seqs
  max_grad_norm: float         = 1.0
  warmup_steps: int          = 1
  num_train_steps: int       = 120 # total gradient steps
  lr_scheduler: str          = "cosine"

  # Reward weights (ablatable)
  correctness_weight: float   = 1.0
  format_reward_weight: float = 0.02 # per tag/format element
  empty_think_penalty: float  = -0.5
  repetition_penalty: float    = -0.2
  length_penalty_weight: float = 0.05
  max_think_words: int        = 500
  missing_answer_penalty: float = -0.5

  # LoRA
  use_lora: bool              = True
  lora_rank: int              = 8
  lora_alpha: int             = 16

  # Data
  data_dir: Optional[str]     = None # if None, build-on-the-fly
  use_math_dataset: bool      = True
  math_max_level: int         = 3
  val_size: int               = 20

  # Logging / saving
  log_every: int              = 1
  eval_every: int             = 999999
  save_every: int             = 20
  wandb_project: Optional[str] = None
  seed: int                   = 42

In [13]:
import os
import json
import math
import random
import argparse
import time
from pathlib import Path
from typing import Optional
from dataclasses import dataclass, asdict

import torch
import torch.nn.functional as F
import numpy as np
import wandb
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk


# Model utilities
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
PEFT_AVAILABLE = True

def load_model_and_ref(config: GRPOConfig):
  """
    Correct loading sequence:

      Step A: Load base model weights (Qwen/Qwen2.5-1.5B)
      Step B: Load Phase 2 SFT LoRA adapter on top → PeftModel
      Step C: Merge LoRA into base weights → plain AutoModelForCausalLM
              This is the SFT-merged model.
      Step D: Clone the SFT-merged model as the FROZEN reference model.
              KL penalty is now measured against SFT behaviour, not the raw base.
      Step E: Apply a fresh GRPO LoRA adapter to the policy (the merged model).
              Only the new LoRA parameters are trainable.

    """
  dtype = torch.bfloat16
  sft_ckpt = Path(config.sft_checkpoint)

  print(f"Loading policy model: {config.model_name}")
  tokenizer = AutoTokenizer.from_pretrained(
      #str(sft_ckpt), # use SFT tokenizer (may have added tokens)
      # Removed config.model_name to avoid multiple 'vocab' arguments
      config.model_name,
      trust_remote_code=True,
      padding_side="left",
  )

  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

  policy = AutoModelForCausalLM.from_pretrained(
      config.model_name,
      dtype=dtype,
      trust_remote_code=True,
  )

  policy.resize_token_embeddings(len(tokenizer))

  sft_peft_model = PeftModel.from_pretrained(
      policy,
      str(sft_ckpt),
      is_trainable=False, # will merge, not continue training
      device_map="auto",
      trust_remote_code=True,
  )

  # Merge SFT LoRA → get plain model with SFT knowledge
  print("Step 3: Merging SFT LoRA weights into base model...")
  sft_merged = sft_peft_model.merge_and_unload()
  sft_merged = sft_merged.to("cuda")
  # sft_merged is now a plain AutoModelForCausalLM with SFT weights baked in

  #sft_merged.save_pretrained(str(sft_ckpt), safe_serialization=True)

  from transformers import BitsAndBytesConfig

  bnb_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.bfloat16,
      bnb_4bit_use_double_quant=True,
  )

  ref_base = AutoModelForCausalLM.from_pretrained(
      config.model_name,
      quantization_config=bnb_config,
      device_map="auto",
      trust_remote_code=True,
  )

  ref_base.resize_token_embeddings(len(tokenizer))  # add here

  ref_peft = PeftModel.from_pretrained(
      ref_base,
      str(sft_ckpt),
      is_trainable=False,
  )

  # Clone as frozen reference
  print("Step 4: Cloning merged model as frozen reference...")
  #import copy
  #ref_model = copy.deepcopy(sft_merged)
  ref_model = ref_peft
  ref_model.eval()
  for p in ref_model.parameters():
    p.requires_grad_(False)

  # Apply fresh GRPO LoRA to policy
  if config.use_lora:
    assert PEFT_AVAILABLE, "Install peft: pip install peft"
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    policy = get_peft_model(sft_merged, lora_cfg)
    policy = policy.to("cuda")
    policy.print_trainable_parameters()
  else:
    policy = sft_merged
    policy.gradient_checkpointing_enable()

  total_policy = sum(p.numel() for p in policy.parameters()) / 1e9
  total_ref = sum(p.numel() for p in ref_model.parameters()) / 1e9
  print(f"\n  Policy:    {total_policy:.2f}B params total")
  print(f"  Reference: {total_ref:.2f}B params (frozen SFT weights)")

  return policy, ref_model, tokenizer


In [14]:
# Generation
def generate_group(
    model,
    tokenizer, # Fixed: changed 'tokenzier' to 'tokenizer'
    prompts: list[str],
    num_generations: int,
    temperature: float,
    top_p: float,
    max_new_tokens: int,
) -> tuple[list[list[str]], list[torch.Tensor], list[int]]:
  """
  For each prompt, sample 'num_generations' completions per prompt
  Returns:
    completions: list[list[str]] - shape[B, G]
    input_lengths: list[int] - prompt token lengths (for masking)
    output_ids: list[tensor] - full token ids for log-prob computation
  """
  model.eval()
  all_completions = []
  all_input_lengths = []
  all_output_ids = []

  for prompt in prompts:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to("cuda")

    input_len = inputs["input_ids"].shape[1]
    all_input_lengths.append(input_len)

    out_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_generations,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.1,
        no_repeat_ngram_size=8,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    all_output_ids.append(out_ids)

    # Decode only new tokens
    new_token_ids = out_ids[:, input_len:]
    decoded = tokenizer.batch_decode(
        new_token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,)
    # Force <thing> prefix
    decoded = ["<think>\n" + d for d in decoded]

    all_completions.append(decoded) # list of G strings

  return all_completions, all_input_lengths, all_output_ids

In [15]:
# Log-probability computation
def _compute_log_probs(
    model,
    output_ids: torch.Tensor,
    input_length: int,
    pad_token_id: int,
    chunk_size=64
) -> torch.Tensor:
  """
  Compute per-token log probabiliteis for the completion portion only.

  Args:
    output_ids: [G, seq_len] token ids (prompt + calculation)
    input_length: number of prompt tokens to mask out

  Returns:
    log_probs: [G] - mean log prob per completion
  """
  outputs = model(output_ids)
  logits = outputs.logits

  shift_logits = logits[:, :-1, :]
  shift_labels = output_ids[:, 1:]

  completion_start = input_length - 1
  shift_logits = shift_logits[:, completion_start:, :]
  shift_labels = shift_labels[:, completion_start:]

  token_log_probs = []

  T = shift_logits.shape[1]

  for start in range(0, T, chunk_size):
      end = min(start + chunk_size, T)

      logits_chunk = shift_logits[:, start:end, :]
      labels_chunk = shift_labels[:, start:end]

      log_probs_chunk = F.log_softmax(logits_chunk, dim=-1)
      gathered = log_probs_chunk.gather(
          2, labels_chunk.unsqueeze(-1)
      ).squeeze(-1)

      token_log_probs.append(gathered)

  token_log_probs = torch.cat(token_log_probs, dim=1)

  pad_mask = shift_labels.ne(pad_token_id).float()

  n_tokens = pad_mask.sum(dim=-1).clamp(min=1)
  return token_log_probs.mul(pad_mask).sum(dim=-1) / n_tokens
  '''
  # Token-level log probs
  log_probs_all = F.log_softmax(shift_logits, dim=-1) # [G, comp_len, vocab]
  token_log_probs = log_probs_all.gather(
      2, shift_labels.unsqueeze(-1)
  ).squeeze(-1) # [G, comp_len]

  # Mask padding tokens
  pad_mask = (shift_labels != 0).float()
  token_log_probs = token_log_probs * pad_mask

  # Mean over completion tokens (ignore padding)
  n_tokens = pad_mask.sum(dim=-1).clamp(min=1)
  mean_log_probs = token_log_probs.sum(dim=-1) / n_tokens #[G]

  return mean_log_probs
  '''

def compute_policy_log_probs(policy, output_ids: torch.Tensor,
                             input_length: int, pad_token_id: int) -> torch.Tensor:
    """
    Policy forward pass - gradients ENABLED (used in training loop)
    """
    return _compute_log_probs(policy, output_ids, input_length, pad_token_id)

@torch.no_grad()
def compute_ref_log_probs(ref_model, output_ids, input_length, pad_token_id):
    ref_device = next(ref_model.parameters()).device
    output_ids = output_ids.to(ref_device)
    ref_lp = _compute_log_probs(ref_model, output_ids, input_length, pad_token_id)
    return ref_lp.to("cuda")

In [16]:
# GRPO Loss
def grpo_loss(
    policy_log_probs: torch.Tensor, # [G]
    ref_log_probs: torch.Tensor, # [G]
    advantages: torch.Tensor, #[G]
    kl_coeff: float,
    clip_eps: float,
) -> tuple[torch.Tensor, dict]:
  """
  GRPO objective:
    L = -E[ min(r*A, clip(r, 1-eps, 1+eps)*A) ] + beta * KL(policy || ref)

    where r = exp(log_pi - log_pi_old) ≈ exp(log_pi - log_ref) at step 0.

  Since we recompute log probs fresh each step (on-policy), we use log_ref as the
  "old" policy approximation for the ratio.
  """
  # Probability ratio (importance weight)
  log_ratio = policy_log_probs - ref_log_probs.detach()
  ratio = torch.exp(log_ratio)

  # Clipped surrogate
  surr1 = ratio * advantages
  surr2 = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * advantages
  policy_loss = -torch.mean(torch.min(surr1, surr2))

  # KL divergence penalty: KL(policy || ref) = log_ratio (first-order approx)
  kl = (ratio - 1 - log_ratio).mean() # unbiased KL estimator
  kl_loss = kl_coeff * kl

  total_loss = policy_loss + kl_loss

  return total_loss, {
      "loss_policy": policy_loss.item(),
      "loss_kl": kl_loss.item(),
      "loss_total": total_loss.item(),
      "mean_ratio": ratio.mean().item(),
      "mean_kl": kl.item(),
      "clip_fraction": ((ratio < 1 - clip_eps) | (ratio > 1 + clip_eps)).float().mean().item(),
  }

In [17]:
# Advantage computation
def compute_advantages(rewards: list[float]) -> torch.Tensor:
  """
  GRPO advantage = group-normalized reward.
    A_i = (r_i - mean(r)) / (std(r) + eps)

  If all rewards in the group are identical (e.g. all wrong),
  advantages are all zero -> no gradient update which is the right behavior
  """
  r = torch.tensor(rewards, dtype=torch.float32)
  mean_r = r.mean()
  std_r = r.std()
  advantages = (r-mean_r) / (std_r + 1e-8)
  return advantages

In [18]:
# Learning rate scheduler
def get_lr(step: int, config: GRPOConfig) -> float:
  """
  Cosine schedule with linear warmup
  """
  if step < config.warmup_steps:
    return config.learning_rate * step / config.warmup_steps
  progress = (step - config.warmup_steps) / max(1, config.num_train_steps - config.warmup_steps)
  return config.learning_rate * 0.5 * (1 + math.cos(math.pi * progress))


In [19]:
# Evaluation
@torch.no_grad()
def evaluate(
    policy,
    tokenizer,
    val_dataset,
    config: GRPOConfig,
    n_eval: int = 100,
) -> dict:
    """
    Greedy decode (pass@1) on val set. Returns accuracy and think-tag metrics.
    """
    policy.eval()

    import re
    #from grpo_rewards import extract_answer, math_equal

    indices = random.sample(range(len(val_dataset)), min(n_eval, len(val_dataset)))
    correct = 0
    think_rate = 0
    think_lengths = []

    for idx in indices:
        ex = val_dataset[idx]
        inputs = tokenizer(
            ex["prompt"],
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(policy.device)

        out = policy.generate(
            **inputs,
            max_new_tokens=config.max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=8,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        new_tokens = out[0, inputs["input_ids"].shape[1]:]
        completion = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,)

        completion = "<think>\n" + completion

        pred = extract_answer(completion)
        if math_equal(pred, ex["gold_answer"]):
            correct += 1

        has_think = "<think>" in completion and "</think>" in completion
        think_rate += has_think
        if has_think:
            m = re.search(r'<think>(.*?)</think>', completion, re.DOTALL)
            think_lengths.append(len(m.group(1).split()) if m else 0)

    n = len(indices)
    policy.train()
    return {
        "eval/pass_at_1":        correct / n,
        "eval/think_tag_rate":   think_rate / n,
        "eval/mean_think_words": float(np.mean(think_lengths)) if think_lengths else 0.0,
        "eval/n":                n,
    }

In [20]:
!pip install -U bitsandbytes>=0.46.1

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    dtype=torch.bfloat16,
    trust_remote_code=True,
).to("cuda")

sft_model = PeftModel.from_pretrained(
    base,
    ".",  # your Phase 2 adapter folder
    is_trainable=False,
).to("cuda")

test_prompt = build_grpo_prompt(
    "Janet’s ducks lay 16 eggs per day. She eats 3 and uses 4 for muffins. "
    "She sells the rest for $2 each. How much does she make?"
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

out = sft_model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
    repetition_penalty=1.1,
    no_repeat_ngram_size=8,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

generated = tokenizer.decode(
    out[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(test_prompt + generated)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

<|im_start|>system
You are a math assistant. Solve the problem step by step. Finish with the final answer in \boxed{}.<|im_end|>
<|im_start|>user
Solve: Janet’s ducks lay 16 eggs per day. She eats 3 and uses 4 for muffins. She sells the rest for $2 each. How much does she make?<|im_end|>
<|im_start|>assistant
To solve this problem, we need to follow these steps:

Step 1: Calculate the total number of eggs laid by Janet's ducks.
Janet's ducks lay 16 eggs per duck per day. Since there is only one duck mentioned, we can assume that it lays 16 eggs per day.

Step 2: Subtract the eggs eaten and used for muffins from the total number of eggs laid.
Janet eats 3 eggs and uses 4 eggs for muffins. So, the remaining eggs are:
16 (total eggs) - 3 (eaten) - 4 (used for muffins) = 9 eggs

Step 3: Determine how many eggs are left after subtracting those used for cooking.
After accounting for the eggs eaten and used for muffin-making, Janet has 9 eggs left.

Step 4: Calculate the revenue from selling 

In [23]:
# Main Training Loop
def train(config: GRPOConfig):
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)

    # W&B
    if config.wandb_project:
        wandb.init(
            project=config.wandb_project,
            name=f"grpo-{Path(config.model_name).name}",
            config=asdict(config),
        )

    # Load model + reference
    policy, ref_model, tokenizer = load_model_and_ref(config)

    # Build reward config
    reward_cfg = RewardConfig(
        correctness_weight=config.correctness_weight,
        think_open_weight=config.format_reward_weight,
        think_close_weight=config.format_reward_weight,
        answer_format_weight=config.format_reward_weight,
        empty_think_penalty=config.empty_think_penalty,
        repetition_penalty=config.repetition_penalty,
        length_penalty_weight=config.length_penalty_weight,
        max_think_words=config.max_think_words,
        missing_answer_penalty=config.missing_answer_penalty,
    )

    # Load dataset
    if config.data_dir and Path(config.data_dir).exists():
        print(f"\nLoading GRPO dataset from {config.data_dir}")
        train_ds = load_from_disk(str(Path(config.data_dir) / "train"))
        val_ds   = load_from_disk(str(Path(config.data_dir) / "val"))
    else:
        print("\nBuilding GRPO dataset...")
        train_ds, val_ds = build_grpo_dataset(
            tokenizer=tokenizer,
            use_gsm8k=True,
            use_math=config.use_math_dataset,
            math_max_level=config.math_max_level,
            val_size=config.val_size,
            output_dir=config.data_dir,
        )

    print(f"  Train: {len(train_ds)}, Val: {len(val_ds)}")

    # Optimizer
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, policy.parameters()),
        lr=config.learning_rate,
        weight_decay=0.01,
        eps=1e-8,
    )

    # Output dir
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save config
    with open(output_dir / "grpo_config.json", "w") as f:
        json.dump(asdict(config), f, indent=2)

    # Shuffle training data into a queue
    train_indices = list(range(len(train_ds)))
    random.shuffle(train_indices)
    idx_cursor = 0

    def next_batch(batch_size: int) -> list[dict]:
        """Pull next batch from shuffled dataset, re-shuffling when exhausted."""
        nonlocal idx_cursor, train_indices
        if idx_cursor + batch_size > len(train_indices):
            random.shuffle(train_indices)
            idx_cursor = 0
        batch_idx = train_indices[idx_cursor: idx_cursor + batch_size]
        idx_cursor += batch_size
        return [train_ds[i] for i in batch_idx]

    # Metric accumulators
    accum_loss     = []
    accum_reward   = []
    accum_kl       = []
    accum_correct  = []
    accum_fmt      = []
    accum_hacking  = []

    policy.train()
    optimizer.zero_grad()
    global_step = 0
    accum_steps = 0

    print(f"\nStarting GRPO training for {config.num_train_steps} steps")
    print(f"  G={config.num_generations}  grad_accum={config.gradient_accumulation}")
    print(f"  SFT checkpoint: {config.sft_checkpoint}")
    print(f"  KL coeff: {config.kl_coeff}  clip_eps: {config.clip_eps}\n")

    pbar = tqdm(total=config.num_train_steps, desc="GRPO")

    while global_step < config.num_train_steps:
        # -----------------------------------------------------------------------
        # 1. Sample a batch of prompts
        # -----------------------------------------------------------------------
        batch = next_batch(config.per_device_batch_size)
        prompts      = [ex["prompt"]      for ex in batch]
        gold_answers = [ex["gold_answer"] for ex in batch]

        # -----------------------------------------------------------------------
        # 2. Generate G completions per prompt
        # -----------------------------------------------------------------------
        completions_batch, input_lengths, output_ids_batch = generate_group(
            model=policy,
            tokenizer=tokenizer,
            prompts=prompts,
            num_generations=config.num_generations,
            temperature=config.temperature,
            top_p=config.top_p,
            max_new_tokens=config.max_new_tokens,
        )

        # -----------------------------------------------------------------------
        # 3. Compute rewards + loss per prompt-group for all completions
        # -----------------------------------------------------------------------
        step_loss = torch.tensor(0.0, device=policy.device)
        step_metrics = {
            "reward_total": [], "reward_correctness": [], "reward_format": [],
            "loss_total": [], "loss_kl": [], "loss_policy": [],
            "clip_fraction": [], "mean_kl": [],
        }
        hacking_flags = []

        policy.train()

        for b_idx, (completions, gold, input_len, out_ids) in enumerate(
            zip(completions_batch, gold_answers, input_lengths, output_ids_batch)
        ):
            # Reward computation (no grad needed)
            rewards, breakdowns = batch_reward_fn(
                completions=completions,
                gold_answers=[gold] * config.num_generations,
                config=reward_cfg,
            )

            # Hacking detection
            hack_info = detect_hacking(completions, rewards)
            hacking_flags.append(hack_info["hacking_suspected"])

            # Advantages via group normalization
            advantages = compute_advantages(rewards).to(policy.device)  # [G]

            # Log-probs under policy (with grad) and reference (no grad)
            out_ids = out_ids.to(policy.device)

            policy_lp = compute_policy_log_probs(
                policy,
                out_ids,
                input_len,
                tokenizer.pad_token_id,
            )   # [G] grad ON
            ref_lp    = compute_ref_log_probs(
                ref_model,
                out_ids,
                input_len,
                tokenizer.pad_token_id) # [G] grad OFF

            # GRPO loss for this prompt-group
            loss, loss_info = grpo_loss(
                policy_log_probs=policy_lp,
                ref_log_probs=ref_lp,
                advantages=advantages.to(policy.device),
                kl_coeff=config.kl_coeff,
                clip_eps=config.clip_eps,
            )

            # Accumulate (normalize by accumulation steps)
            loss = loss / config.gradient_accumulation
            loss.backward()
            step_loss = step_loss + loss.detach()

            # Collect metrics
            step_metrics["reward_total"].append(float(np.mean(rewards)))
            step_metrics["reward_correctness"].append(
                float(np.mean([b["reward_correctness"] for b in breakdowns]))
            )
            step_metrics["reward_format"].append(
                np.mean([b["reward_format"] for b in breakdowns])
            )
            for k in ["loss_total", "loss_kl", "loss_policy", "clip_fraction", "mean_kl"]:
                step_metrics[k].append(loss_info[k])

            # Clear cache
            del rewards, breakdowns, advantages
            del policy_lp, ref_lp, loss, loss_info, out_ids
            torch.cuda.empty_cache()

        #del completions_batch, input_lengths, output_ids_batch
        #torch.cuda.empty_cache()
        # -----------------------------------------------------------------------
        # 4. Optimizer step (after gradient_accumulation prompts)
        # -----------------------------------------------------------------------
        accum_steps += 1
        if accum_steps % config.gradient_accumulation == 0:
            # Update learning rate
            lr = get_lr(global_step, config)
            for pg in optimizer.param_groups:
                pg["lr"] = lr

            # Clip gradients
            grad_norm = torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, policy.parameters()),
                config.max_grad_norm,
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            global_step += 1
            pbar.update(1)

            # Accumulate for logging
            accum_loss.append(step_loss.item())
            accum_reward.append(float(np.mean(step_metrics["reward_total"])))
            accum_kl.append(float(np.mean(step_metrics["mean_kl"])))
            accum_correct.append(float(np.mean(step_metrics["reward_correctness"])))
            accum_fmt.append(float(np.mean(step_metrics["reward_format"])))
            accum_hacking.append(float(any(hacking_flags)))

            # -----------------------------------------------------------------------
            # 5. Logging
            # -----------------------------------------------------------------------
            if global_step % config.log_every == 0:
                log = {
                    "train/step":              global_step,
                    "train/loss":              np.mean(accum_loss),
                    "train/reward_mean":       np.mean(accum_reward),
                    "train/reward_correct":    np.mean(accum_correct),
                    "train/reward_format":     np.mean(accum_fmt),
                    "train/kl_divergence":     np.mean(accum_kl),
                    "train/grad_norm":         grad_norm.item(),
                    "train/learning_rate":     lr,
                    "train/loss_policy":       np.mean(step_metrics["loss_policy"]),
                    "train/clip_fraction":     np.mean(step_metrics["clip_fraction"]),
                    "train/hacking_rate":      np.mean(accum_hacking),
                }

                train_log_path = output_dir / "train_metrics.jsonl"
                # Saving training files
                with open(train_log_path, "a") as f:
                    f.write(json.dumps(log) + "\n")

                pbar.set_postfix({
                    "loss":    f"{log['train/loss']:.4f}",
                    "reward":  f"{log['train/reward_mean']:.3f}",
                    "correct": f"{log['train/reward_correct']:.3f}",
                    "kl":      f"{log['train/kl_divergence']:.4f}",
                })
                if config.wandb_project and WANDB_AVAILABLE:
                    wandb.log(log)

                # Reset accumulators
                accum_loss.clear(); accum_reward.clear(); accum_kl.clear()
                accum_correct.clear(); accum_fmt.clear(); accum_hacking.clear()

            # -----------------------------------------------------------------------
            # 6. Evaluation
            # -----------------------------------------------------------------------
            if global_step % config.eval_every == 0:
                print(f"\nEvaluating at step {global_step}...")
                eval_metrics = evaluate(policy, tokenizer, val_ds, config, n_eval=100)
                eval_metrics["eval/step"] = global_step
                print(
                    f"  Pass@1: {eval_metrics['eval/pass_at_1']:.3f} | "
                    f"  <think> rate: {eval_metrics['eval/think_tag_rate']:.3f} | "
                    f"  Avg think words: {eval_metrics['eval/mean_think_words']:.0f}"
                )
                if config.wandb_project:
                    wandb.log(eval_metrics)

                # Save eval results
                eval_path = output_dir / f"eval_step_{global_step}.json"
                with open(eval_path, "w") as f:
                    json.dump(eval_metrics, f, indent=2)

            # -----------------------------------------------------------------------
            # 7. Checkpoint saving
            # -----------------------------------------------------------------------
            if global_step % config.save_every == 0:
                ckpt_dir = output_dir / f"checkpoint-{global_step}"
                print(f"\nSaving checkpoint \u2192 {ckpt_dir}")
                policy.save_pretrained(str(ckpt_dir))
                tokenizer.save_pretrained(str(ckpt_dir))

            # Free large tensors from this completed optimizer step
            del completions_batch, output_ids_batch, step_loss
            torch.cuda.empty_cache()

    pbar.close()

    # ---------------------------------------------------------------------------
    # Final save
    # ---------------------------------------------------------------------------
    print(f"\nTraining complete. Saving final model...")
    final_dir = output_dir / "final"
    policy.save_pretrained(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    # If LoRA, merge weights into base model for clean evaluation
    if config.use_lora:
        print("Merging GRPO LoRA into SFT-merged weights...")
        merged_dir = output_dir / "merged"
        merged = policy.merge_and_unload()
        merged.save_pretrained(str(merged_dir))
        tokenizer.save_pretrained(str(merged_dir))
        print(f"Final merged model -> {merged_dir}")
        print("Use the merged/ directory for Phase 4 evaluation.")

    if config.wandb_project and WANDB_AVAILABLE:
        wandb.finish()

    return output_dir


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Phase 3: GRPO Training")

    # Model
    # Phase 2 SFT adapter
    parser.add_argument("--sft_checkpoint", type=str, required=True, help="Path to Phase 2 SFT LoRA adapter directory "
                             "(must contain adapter_config.json)")
    parser.add_argument("--base_model",          type=str, default="Qwen/Qwen2.5-1.5B",
                        help="SFT checkpoint path or HF model name")
    parser.add_argument("--output_dir",     type=str, default="./checkpoints/grpo")

    # GRPO hyperparameters
    parser.add_argument("--num_generations",type=int,   default=8)
    parser.add_argument("--temperature",    type=float, default=0.9)
    parser.add_argument("--kl_coeff",       type=float, default=0.01)
    parser.add_argument("--clip_eps",       type=float, default=0.2)
    parser.add_argument("--max_new_tokens", type=int,   default=512)

    # Optimization
    parser.add_argument("--learning_rate",  type=float, default=1e-6)
    parser.add_argument("--batch_size",     type=int,   default=1)
    parser.add_argument("--grad_accum",     type=int,   default=16)
    parser.add_argument("--num_steps",      type=int,   default=1000)
    parser.add_argument("--warmup_steps",   type=int,   default=20)

    # Reward ablation flags
    parser.add_argument("--correctness_weight",   type=float, default=1.0)
    parser.add_argument("--format_reward_weight", type=float, default=0.3)
    parser.add_argument("--empty_think_penalty",  type=float, default=-0.5)
    parser.add_argument("--repetition_penalty",   type=float, default=-0.2)
    parser.add_argument("--missing_answer_penalty", type=float, default=-0.3)
    parser.add_argument("--kl_coeff_ablate",      type=float, default=None)

    # LoRA
    parser.add_argument("--use_lora",       type=lambda x: x.lower() == "true", default=True)
    parser.add_argument("--lora_rank",      type=int, default=64)

    # Data
    parser.add_argument("--data_dir",       type=str, default=None)
    parser.add_argument("--no_math",        action="store_true")

    # Logging
    parser.add_argument("--log_every",      type=int, default=10)
    parser.add_argument("--eval_every",     type=int, default=50)
    parser.add_argument("--save_every",     type=int, default=100)
    parser.add_argument("--wandb_project",  type=str, default=None)
    parser.add_argument("--seed",           type=int, default=42)

    # Provide a placeholder value for sft_checkpoint since it's required
    # The actual adapter files are in the current directory, so we point to '.'
    args = parser.parse_args(["--sft_checkpoint", "."])

    config = GRPOConfig(
        sft_checkpoint=args.sft_checkpoint,
        model_name=args.base_model, # Changed from base_model_name to model_name
        output_dir=args.output_dir,
        num_generations=args.num_generations,
        temperature=args.temperature,
        kl_coeff=args.kl_coeff_ablate if args.kl_coeff_ablate is not None else args.kl_coeff,
        clip_eps=args.clip_eps,
        max_new_tokens=args.max_new_tokens,
        learning_rate=args.learning_rate,
        per_device_batch_size=args.batch_size,
        gradient_accumulation=args.grad_accum,
        num_train_steps=args.num_steps,
        warmup_steps=args.warmup_steps,
        correctness_weight=args.correctness_weight,
        format_reward_weight=args.format_reward_weight,
        empty_think_penalty=args.empty_think_penalty,
        repetition_penalty=args.repetition_penalty,
        use_lora=args.use_lora,
        lora_rank=args.lora_rank,
        data_dir=args.data_dir,
        use_math_dataset=not args.no_math,
        log_every=args.log_every,
        eval_every=args.eval_every,
        save_every=args.save_every,
        wandb_project=args.wandb_project,
        seed=args.seed,
        missing_answer_penalty=args.missing_answer_penalty,
    )

    train(config)

#if __name__ == "__main__":
#   main()


In [ ]:
config = GRPOConfig(
    sft_checkpoint=".",
    model_name="Qwen/Qwen2.5-1.5B",
    output_dir="./checkpoints/grpo_final_clean_v2",

    num_generations=3,
    max_new_tokens=256,
    gradient_accumulation=3,
    num_train_steps=120,

    per_device_batch_size=1,
    learning_rate=5e-7,
    warmup_steps=8,

    eval_every=60,
    save_every=120,
    log_every=1,

    kl_coeff=0.02,
    clip_eps=0.2,

    correctness_weight=1.0,
    format_reward_weight=0.2,
    empty_think_penalty=-0.1,
    repetition_penalty=-0.3,
    missing_answer_penalty=-0.3,

    use_lora=True,
    lora_rank=12,
    lora_alpha=24,

    use_math_dataset=False, # Since only evaluating on GSM8k
    val_size=100,
    seed=42,
)

train(config)

Loading policy model: Qwen/Qwen2.5-1.5B


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step 3: Merging SFT LoRA weights into base model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step 4: Cloning merged model as frozen reference...
trainable params: 13,848,576 || all params: 1,557,146,624 || trainable%: 0.8894

  Policy:    1.56B params total
  Reference: 0.91B params (frozen SFT weights)

Building GRPO dataset...
Loading GSM8K...


...

  GSM8K: 7473 examples

GRPO dataset — Train: 7373, Val: 100
  Train: 7373, Val: 100

Starting GRPO training for 120 steps
  G=3  grad_accum=3
  SFT checkpoint: .
  KL coeff: 0.02  clip_eps: 0.2



GRPO:  50%|█████     | 60/120 [56:51<52:49, 52.82s/it, loss=0.0315, reward=0.533, correct=0.333, kl=0.0186]


Evaluating at step 60...
  Pass@1: 0.160 |   <think> rate: 0.000 |   Avg think words: 0


GRPO: 100%|██████████| 120/120 [2:14:42<00:00, 59.32s/it, loss=0.0140, reward=0.433, correct=0.333, kl=0.0145]


Evaluating at step 120...


In [25]:
import re
import torch
from decimal import Decimal, InvalidOperation
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "checkpoints/grpo_final_clean_v2/merged"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()


def normalize_num(x):
    """
    Normalize numeric answers like:
    '460.' -> '460'
    '$1,200' -> '1200'
    '18 dollars' -> '18'
    """
    if x is None:
        return None

    x = str(x).replace(",", "").replace("$", "").strip()

    # keep only first valid number
    nums = re.findall(r"-?\d+(?:\.\d+)?", x)
    if not nums:
        return None

    x = nums[-1]

    try:
        d = Decimal(x)
        return str(d.normalize())
    except InvalidOperation:
        return x.rstrip(".")


def extract_gsm8k_answer(answer_text):
    """
    GSM8K answers usually contain #### final_answer
    """
    if "####" in answer_text:
        return normalize_num(answer_text.split("####")[-1])

    return normalize_num(answer_text)


def extract_model_answer(output_text):
    if output_text is None:
        return None

    # 1. FIRST: boxed (your training format)
    boxed = re.findall(r"\\boxed\{([^}]+)\}", output_text)
    if boxed:
        return normalize_num(boxed[-1])

    # 2. fallback
    answer_match = re.search(
        r"<answer>(.*?)</answer>",
        output_text,
        re.DOTALL | re.IGNORECASE,
    )
    if answer_match:
        return normalize_num(answer_match.group(1))

    # 3. final fallback
    return normalize_num(output_text)

'''
def build_prompt(question):
    return f"""Solve the following math problem step by step.

Question:
{question}

Give your final answer in this exact format:
<answer>final answer</answer>
"""
'''


def generate_response(prompt, max_new_tokens=768):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=8,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if decoded.startswith(prompt):
        decoded = decoded[len(prompt):].strip()

    return decoded


dataset = load_dataset("gsm8k", "main", split="test")

num_examples = 10
correct = 0

for i in range(num_examples):
    ex = dataset[i]

    question = ex["question"]
    gold_raw = ex["answer"]
    gold = extract_gsm8k_answer(gold_raw)

    prompt = build_grpo_prompt(question, tokenizer)
    output = generate_response(prompt)

    pred = extract_model_answer(output)
    is_correct = math_equal(pred, gold) #pred == gold

    if is_correct:
        correct += 1

    print("\n" + "=" * 80)
    print(f"Example {i + 1}")
    print("=" * 80)

    print("\nQUESTION:")
    print(question)

    print("\nMODEL OUTPUT:")
    print(output)

    print("\nGOLD RAW ANSWER:")
    print(gold_raw)

    print("\nNORMALIZED GOLD ANSWER:")
    print(gold)

    print("\nEXTRACTED MODEL ANSWER:")
    print(pred)

    print("\nCORRECT?")
    print(is_correct)

print("\n" + "=" * 80)
print(
    f"Accuracy on {num_examples} examples: "
    f"{correct}/{num_examples} = {correct / num_examples:.2%}"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]


Example 1

QUESTION:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

MODEL OUTPUT:
system
You are a math assistant. Solve the problem step by step. Finish with the final answer in \boxed{}.
user
Solve: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
assistant
Solve: The number of ducks that Marvin has is 3 times as many as the number of geese he owns. If there are 50 geese, how many ducks and geese does Marvin have altogether?骟
骟user
Solve: A farmer harvested 48 apples from his orchard. He sold half of them to a local store an